In [11]:
!pip install langchain-chroma

In [1]:
import os
import json
import requests
import re
from langchain_chroma import Chroma
from langchain_community.document_loaders import JSONLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

/home/eslee03/마음돌봄/.venv-ros/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# ================== 1. 데이터 크롤링 및 정제 ==================

url = "https://openapi.naver.com/v1/search/blog.json"
word = "대학병원" + "건강칼럼"

all_results = []
headers = {
    "X-Naver-Client-Id": "NXvICODmXm4RIOHse8dd",
    "X-Naver-Client-Secret": "yxJ9c68Szf"
}

# 1,000개 수집 (100개씩 10번)
for start_index in range(1, 1001, 100):
    params = {
        "query": word,
        "display": 100,
        "start": start_index,
        "sort": "sim"
    }
    res = requests.get(url, params=params, headers=headers)
    
    if res.status_code == 200:
        items = res.json().get('items', [])
        all_results.extend(items)
    else:
        print(f"Error: {res.status_code}")
        break

def clean_html(raw_html):
    cleanr = re.compile('<.*?>')
    return re.sub(cleanr, '', raw_html)

# ================== 2. JSONL 파일 저장 ==================

# med_data 폴더 생성
save_dir = "./med_data"
os.makedirs(save_dir, exist_ok=True)

# 저장할 jsonl 파일 경로
jsonl_file_path = os.path.join(save_dir, "health_info_blog_data.jsonl")

# 정제 후 jsonl로 저장
with open(jsonl_file_path, "w", encoding="utf-8") as f:
    for item in all_results:
        data_dict = {
            "title": clean_html(item.get('title', '')),
            "description": clean_html(item.get('description', '')),
            "link": item.get('link', '')
        }
        f.write(json.dumps(data_dict, ensure_ascii=False) + "\n")

print(f"{jsonl_file_path} 에 크롤링 데이터 저장 완료")

./med_data/health_info_blog_data.jsonl 에 크롤링 데이터 저장 완료
